# 05.5 — Beam Search

**Problem it solves:** Greedy decoding picks the single best token at each step. This can lead to locally good but globally bad sequences. Beam search explores top-k partial sequences simultaneously.

**The tradeoff:** Larger beam width → better quality but exponential compute. In practice, beam width 4-8 gives most of the quality gain.

---

In [ ]:
import numpy as np
import heapq
from dataclasses import dataclass, field

@dataclass(order=True)
class Beam:
    score: float          # negative log prob (for min-heap)
    tokens: list = field(compare=False)
    state: object = field(compare=False, default=None)

def beam_search(start_token: int, eos_token: int,
                next_token_logprobs_fn,   # fn(tokens, state) -> (log_probs, new_state)
                beam_width: int = 4,
                max_len: int = 20,
                length_penalty: float = 0.6) -> list:
    """
    Beam search decoder.
    
    length_penalty: penalize shorter sequences (Wu et al., 2016)
    score = log_prob / (5 + len)^alpha / (5+1)^alpha
    """
    
    # Initialize: one beam with the start token
    initial_log_probs, initial_state = next_token_logprobs_fn([start_token], None)
    
    # Top-k initial beams
    top_k = np.argsort(initial_log_probs)[::-1][:beam_width]
    beams = [
        Beam(score=-initial_log_probs[tok],  # negate for min-heap
             tokens=[start_token, tok],
             state=initial_state)
        for tok in top_k
    ]
    
    completed = []
    
    for step in range(max_len - 1):
        if len(completed) >= beam_width:
            break
        
        candidates = []
        
        for beam in beams:
            if beam.tokens[-1] == eos_token:
                # Apply length penalty
                lp = ((5 + len(beam.tokens)) / 6) ** length_penalty
                normalized_score = beam.score / lp
                completed.append(Beam(normalized_score, beam.tokens, None))
                continue
            
            log_probs, new_state = next_token_logprobs_fn(beam.tokens, beam.state)
            
            # Expand: try all next tokens
            for tok in range(len(log_probs)):
                new_score = beam.score - log_probs[tok]  # accumulate neg log prob
                candidates.append(Beam(
                    score=new_score,
                    tokens=beam.tokens + [tok],
                    state=new_state
                ))
        
        # Keep top-k candidates
        candidates.sort()
        beams = candidates[:beam_width]
    
    # Add remaining unfinished beams
    completed.extend(beams)
    completed.sort()
    
    return completed

# Demo: simple toy language model
# Vocab: a=0, b=1, c=2, <EOS>=3
# Transition probs: a->b(0.7)|c(0.3), b->a(0.4)|c(0.6), c-><EOS>(0.8)|a(0.2)

TRANS = {
    0: np.log([0.001, 0.7, 0.3, 0.001]),   # from a
    1: np.log([0.4, 0.001, 0.6, 0.001]),   # from b
    2: np.log([0.2, 0.001, 0.001, 0.8]),   # from c -> EOS
    3: np.log([0.25, 0.25, 0.25, 0.25]),   # from EOS (shouldn't happen)
}

VOCAB = ['a', 'b', 'c', '<EOS>']

def toy_lm(tokens, state):
    """Toy next-token log probability function."""
    last = tokens[-1] if tokens else 0
    return TRANS[last], None

print("Beam search on toy language model:")
print("(Vocab: a=0, b=1, c=2, EOS=3)")
print()

for beam_width in [1, 2, 4]:
    results = beam_search(
        start_token=0,      # start from 'a'
        eos_token=3,
        next_token_logprobs_fn=toy_lm,
        beam_width=beam_width,
        max_len=8
    )
    print(f"Beam width={beam_width}:")
    for r in results[:3]:
        seq = ''.join(VOCAB[t] for t in r.tokens)
        print(f"  score={r.score:.3f}  seq={seq!r}")

In [ ]:
# Beam search vs greedy: where they differ

# Example where greedy fails:
# At step 1: P(A)=0.6, P(B)=0.4
# At step 2 given A: P(good)=0.3, P(bad)=0.7  -> P(A,bad) = 0.6*0.7 = 0.42
#              given B: P(good)=0.9, P(bad)=0.1  -> P(B,good) = 0.4*0.9 = 0.36
# Greedy chooses A then bad = 0.42
# Optimal is A then bad = 0.42  (so greedy wins here)

# But now flip it:
# P(A)=0.4, P(B)=0.6
# Given A: P(great)=0.99  -> P(A,great) = 0.4*0.99 = 0.396
# Given B: P(ok)=0.3      -> P(B,ok) = 0.6*0.3 = 0.18
# Greedy chooses B -> P(B,ok) = 0.18  WRONG
# Beam search explores A -> P(A,great) = 0.396  CORRECT

print("Case where greedy decoding fails:")
print()
print("Step 1 probabilities: A=0.4, B=0.6")
print("Step 2 given A: 'great'=0.99, 'ok'=0.01")
print("Step 2 given B: 'great'=0.01, 'ok'=0.99")
print()
print("Greedy: picks B (0.6 > 0.4), then 'ok' -> P=0.6*0.99=0.594")
print("Wait, greedy actually wins here!")
print()
print("Greedy fails when FIRST steps look bad but lead to globally better sequences.")
print("This happens with: longer context, left-branching grammars, rare but correct words.")

# The length penalty: why it's needed
print()
print("Length penalty (Wu et al. 2016):")
print("Without: beam search prefers shorter sequences (lower cumulative log prob)")
print("With: normalize by length^alpha")
for alpha in [0.0, 0.6, 1.0]:
    short_score = -3.0  # log prob of 3-token sequence
    long_score  = -4.5  # log prob of 6-token sequence (more informative)
    lp_short = ((5 + 3) / 6) ** alpha
    lp_long  = ((5 + 6) / 6) ** alpha
    norm_short = short_score / lp_short
    norm_long  = long_score  / lp_long
    winner = 'short' if norm_short > norm_long else 'long'
    print(f"  alpha={alpha}: short={norm_short:.3f} long={norm_long:.3f} -> {winner} wins")

## Summary

**Beam search tradeoffs:**

| Width | Quality | Speed | Memory |
|-------|---------|-------|--------|
| 1 (greedy) | lowest | fastest | O(1) |
| 4-8 | good | moderate | O(k) |
| 50+ | marginal gain | slow | expensive |

**Practical notes:**
- GPT-style LLMs use sampling (top-k, nucleus/top-p) not beam search for diversity
- Beam search is still standard for machine translation
- Length penalty alpha=0.6 is the Google NMT default

**What breaks beam search:**
- Exposure bias: model was trained with teacher forcing, decoded greedily — distribution mismatch
- Degeneration: without diversity incentives, beams can collapse to near-identical sequences

---
**Module 05 complete. Next: Module 06 — Attention & Transformers**